In [2]:
from pathlib import Path
import shutil

# ==== 設定 ====

# 元フォルダ（~ はホームディレクトリとして解釈）
src_root = Path.home() / "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/STEP3.2_U25_signlessCorr_MDS_20251027"

# 置き換え先の親フォルダ
dst_base = Path.home() / "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters"

# 新しく作るフォルダ名（必要ならここは好みで変更してOK）
dst_root = dst_base / "STEP3.2_U25_signlessCorr_MDS_20251027_OHFPswap"

# 上書き実行するかどうか（False なら実際にコピーする）
DRY_RUN = False   # 最初に挙動を確認したい場合は True にして実行


# ==== ヘルパー ====

def swap_name_in_dataset_subtree(name: str, dataset: str, unchanged_log: set, rel_path_str: str) -> str:
    """
    dataset が 'OH' のとき:
        - 'OH' を含む部分は 'FP' に置換
        - 'FP' を含む場合は「変更しない」＆ rel_path を記録
    dataset が 'FP' のときは逆。
    """
    src_tag = dataset
    dst_tag = "FP" if dataset == "OH" else "OH"

    new_name = name

    if src_tag in name:
        # 通常ケース：大元のフォルダと同じタグを含むので置換対象
        new_name = name.replace(src_tag, dst_tag)
    elif dst_tag in name:
        # 逆側のタグを含む → 手作業修正などの可能性 → 変更せずログに残す
        unchanged_log.add(rel_path_str)

    # どちらも含まれない場合はそのまま
    return new_name


# ==== メイン処理 ====

if not src_root.exists():
    raise SystemExit(f"❌ 元フォルダが存在しません: {src_root}")

print(f"▶ Source: {src_root}")
print(f"▶ Dest  : {dst_root}")
print(f"▶ DRY_RUN = {DRY_RUN}")

unchanged_paths = set()  # 「変更不要」と判断したパスの記録

for path in sorted(src_root.rglob("*")):
    rel = path.relative_to(src_root)
    parts = list(rel.parts)

    if not parts:
        continue

    # 大元のフォルダ名（トップレベル）が OH か FP か判定
    dataset = parts[0] if parts[0] in ("OH", "FP") else None

    new_parts = []

    for i, p in enumerate(parts):
        rel_str = str(rel)
        if dataset in ("OH", "FP"):
            # OH / FP サブツリー内では swap ロジックを適用
            new_p = swap_name_in_dataset_subtree(p, dataset, unchanged_paths, rel_str)
        else:
            # それ以外の階層（TOP など）はそのまま
            new_p = p
        new_parts.append(new_p)

    new_rel = Path(*new_parts)
    dst_path = dst_root / new_rel

    if DRY_RUN:
        # dry-run のときは動作だけ表示
        if path.is_dir():
            print(f"[DIR] {rel} -> {dst_path.relative_to(dst_base)}")
        else:
            print(f"[FILE] {rel} -> {dst_path.relative_to(dst_base)}")
        continue

    # 実コピー
    if path.is_dir():
        dst_path.mkdir(parents=True, exist_ok=True)
    else:
        dst_path.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(path, dst_path)

print("\n✅ コピー処理完了")

# ==== 変更しなかった（= 大元と逆タグを含んでいた）パスの一覧を表示 ====

if unchanged_paths:
    print("\n⚠ 次のパスは、大元フォルダとは逆側のタグを含んでいたため、名前を変更していません。")
    print("   （手作業修正などの可能性があるので要確認）\n")
    for p in sorted(unchanged_paths):
        print(" -", p)
else:
    print("\nℹ 大元のフォルダ名と比較して、変更不要と判断された OH/FP 名を持つパスはありませんでした。")


▶ Source: /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/STEP3.2_U25_signlessCorr_MDS_20251027
▶ Dest  : /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters/STEP3.2_U25_signlessCorr_MDS_20251027_OHFPswap
▶ DRY_RUN = False

✅ コピー処理完了

ℹ 大元のフォルダ名と比較して、変更不要と判断された OH/FP 名を持つパスはありませんでした。
